# 1.11 — Math Class 🧋

### AP CSA · Unit 1: Using Objects and Methods
**Boba Cafe Series · Lesson 11**

---

> **Setup note:** every code cell runs on the **IJava kernel** (Java 17+). Check that the kernel picker says *Java*. Each cell declares a class and then calls it with `ClassName.main(null);`.

> **A small preview:** a few cells below use a `for` loop to run something many times in a row. Loops are the formal subject of **Unit 2** — here one is only used as a tool, to run an experiment thousands of times and check the results. You don't need to understand loops deeply yet to read what these particular ones are doing.

## Every drink at the cafe has been the same drink

Look back across this entire series. Every `main` you've called has produced the exact same output, every single time you ran it. That's normal for a cafe simulation used to *teach* — but it's nothing like a real cafe, where the rush hour crowd is never the same size twice and nobody orders in the same order.

Today that changes. You already know four `Math` methods from scattered use across Lessons 1.5, 1.7, and 1.10 — `sqrt`, `pow`, `abs`, and the constant `MAX_VALUE`. This lesson gives you the complete, official set in one place, and adds the one that actually makes the cafe unpredictable: `Math.random()`.

By the end, Byte-Sized Boba can simulate a rush hour where nobody — not even you, the programmer — knows how many customers are about to walk in.

### What you'll be able to do by the end

| # | Objective | CED reference |
|---|---|---|
| 1 | Use the AP-tested `Math` methods correctly and predict their return types | 1.11.A |
| 2 | State exactly what range `Math.random()` returns | 1.11.A |
| 3 | Scale `Math.random()` into a random integer within any range | 1.11.A |
| 4 | Identify and fix the classic **off-by-one** error in a random-range formula | 1.11.A |
| 5 | Combine casting, ranges, and `Math.random()` to simulate something unpredictable | 1.11.A |

---

## Part 1 — The AP-tested `Math` methods, all in one place

`Math` is a class from `java.lang` (Lesson 1.7 — no `import` needed) made entirely of `static` methods (Lesson 1.10 — called through the class name, no object ever built).

The AP exam draws from a specific, short list. These five entries are the ones on the official **AP Java Quick Reference** — the sheet you're handed during the actual exam:

| Entry | Returns | Description |
|---|---|---|
| `static int abs(int x)` | `int` | Absolute value of `x` |
| `static double abs(double x)` | `double` | Absolute value of `x` |
| `static double pow(double base, double exp)` | `double` | `base` raised to the `exp` power |
| `static double sqrt(double x)` | `double` | Positive square root of `x` |
| `static double random()` | `double` | A value `>= 0.0` and `< 1.0` |

You've used the first four before. `random()` is the new one, and it gets the rest of this lesson.

> **Beyond the exam:** you'll see other `Math` methods in the wild — `Math.round`, `Math.ceil`, `Math.floor`, `Math.min`, `Math.max`. They're genuinely useful and you're welcome to use them, but **they are not on the AP Java Quick Reference**, so the exam won't require them. A short bonus section at the end shows what they do, clearly marked as extra.

In [ ]:
public class MathReview {
    public static void main(String[] args) {
        System.out.println("abs(-7):        " + Math.abs(-7));       // int in, int out
        System.out.println("abs(-7.5):      " + Math.abs(-7.5));     // double in, double out
        System.out.println("pow(2, 10):     " + Math.pow(2, 10));    // ALWAYS a double
        System.out.println("sqrt(81):       " + Math.sqrt(81));      // ALWAYS a double
    }
}

MathReview.main(null);

---

## Part 2 — `Math.random()`

> **`Math.random()` returns a `double` that is `>= 0.0` and `< 1.0`.**

That upper bound matters: it is **strictly less than** `1.0`. `1.0` itself can never come back, no matter how many times you call it. Every value returned lands somewhere in a half-open interval — closed at the bottom, open at the top:

```
   [0.0 ........................................... 1.0)
    ^                                                ^
  possible                                     never possible
```

Run this a few times. You'll get different numbers on each run — that's the entire point.

In [ ]:
public class RandomSample {
    public static void main(String[] args) {
        System.out.println("Five calls to Math.random():");
        System.out.println(Math.random());
        System.out.println(Math.random());
        System.out.println(Math.random());
        System.out.println(Math.random());
        System.out.println(Math.random());
        System.out.println();
        System.out.println("Re-run this cell -- the five numbers above will be different every time.");
    }
}

RandomSample.main(null);

Since the numbers change every run, "print it and eyeball it" only gets you so far. A better way to check a claim about `Math.random()` is to call it thousands of times and see whether every single result obeys the rule.

In [ ]:
public class BoundsCheck {
    public static void main(String[] args) {
        boolean allInBounds = true;
        double observedMin = 2.0;
        double observedMax = -1.0;

        for (int i = 0; i < 500000; i++) {
            double r = Math.random();
            if (r < 0.0 || r >= 1.0) allInBounds = false;
            if (r < observedMin) observedMin = r;
            if (r > observedMax) observedMax = r;
        }

        System.out.println("Every one of 500,000 calls landed in [0.0, 1.0)? " + allInBounds);
        System.out.println("Smallest value seen:  " + observedMin);
        System.out.println("Largest value seen:   " + observedMax);
    }
}

BoundsCheck.main(null);

Half a million calls, every single one inside the promised range, and the largest value observed still fell short of `1.0`. That's `Math.random()`'s contract, proven rather than just stated — the same instinct that mattered back in Lesson 1.8 when you learned to trust a documented postcondition.

---

## Part 3 — Scaling into a range of integers

A `double` between `0.0` and just-under-`1.0` isn't very useful by itself. What the cafe actually needs is things like "a random cup size from 1 to 3" or "a random discount from 10 to 50 percent." Getting there takes three steps, and each one is a rule you already know.

**Step 1 — scale.** Multiplying `Math.random()` by `n` stretches the range from `[0.0, 1.0)` to `[0.0, n)`.

**Step 2 — cast.** `(int)` truncates the result down to a whole number in `{0, 1, 2, ..., n-1}` — this is exactly the truncation rule from Lesson 1.5. Casting never rounds.

**Step 3 — shift.** Adding `min` slides the whole range up so it starts wherever you want instead of at `0`.

```
   (int)(Math.random() * n) + min
        ^^^^^^^^^^^^^^^^^^^   ^^^
         gives 0 .. n-1      shifts the floor to "min"
```

The full formula for an **inclusive** range from `min` to `max`:

```
   (int)(Math.random() * (max - min + 1)) + min
```

That `+ 1` inside the parentheses is not decoration. Without it, `max` can never be produced — and you're about to watch that happen.

In [ ]:
public class DieRoll {
    public static void main(String[] args) {
        int min = 1;
        int max = 6;

        // (max - min + 1) = 6 possible values: 1, 2, 3, 4, 5, 6
        int roll = (int)(Math.random() * (max - min + 1)) + min;

        System.out.println("You rolled: " + roll);
        System.out.println("Re-run this cell for a new roll.");
    }
}

DieRoll.main(null);

Again, one run only proves one value landed somewhere plausible. To actually check that the formula reaches **both** endpoints — never goes below `1`, never goes above `6` — run it enough times to make missing an endpoint astronomically unlikely.

In [ ]:
public class DieRollCheck {
    public static void main(String[] args) {
        int min = 1;
        int max = 6;

        int observedMin = Integer.MAX_VALUE;
        int observedMax = Integer.MIN_VALUE;

        for (int i = 0; i < 200000; i++) {
            int roll = (int)(Math.random() * (max - min + 1)) + min;
            if (roll < observedMin) observedMin = roll;
            if (roll > observedMax) observedMax = roll;
        }

        System.out.println("200,000 rolls of a 1-6 die.");
        System.out.println("Smallest rolled: " + observedMin);
        System.out.println("Largest rolled:  " + observedMax);
    }
}

DieRollCheck.main(null);

`1` and `6`, exactly the boundaries the formula promised — verified across 200,000 rolls rather than taken on faith.

---

## Part 4 — The off-by-one trap

This is the single most common mistake with `Math.random()`, and it produces the quietest kind of bug there is: **the program never crashes, and the output looks completely plausible every single time you glance at it.**

Drop the `+ 1`:

```java
(int)(Math.random() * (max - min)) + min      // MISSING the + 1
```

For a 1–6 die, `max - min` is `5` instead of `6`. That single missing possibility means the formula can only ever land on `1, 2, 3, 4, 5` — **`6` is now mathematically impossible.** No exception gets thrown. No output looks obviously wrong. You could stare at ten rolls in a row and never notice a `6` is missing, because five other faces still show up constantly.

In [ ]:
public class OffByOneBug {
    public static void main(String[] args) {
        int min = 1;
        int max = 6;

        int observedMin = Integer.MAX_VALUE;
        int observedMax = Integer.MIN_VALUE;

        for (int i = 0; i < 200000; i++) {
            int roll = (int)(Math.random() * (max - min)) + min;   // the +1 is MISSING
            if (roll < observedMin) observedMin = roll;
            if (roll > observedMax) observedMax = roll;
        }

        System.out.println("200,000 rolls with the BROKEN formula.");
        System.out.println("Smallest rolled: " + observedMin);
        System.out.println("Largest rolled:  " + observedMax + "   <-- should be 6. It never was.");
    }
}

OffByOneBug.main(null);

Two hundred thousand rolls, and `6` never once appeared. That's not bad luck — it's mathematically impossible with this formula, and it will fail identically **every time** you run it, in every version of Java, forever. This is a **logic error** in its purest form: syntactically perfect, runs flawlessly, produces a wrong answer with total confidence.

**The general lesson, not just about dice:** whenever you build a range with subtraction, stop and ask whether the top end needs a `+ 1` to actually be reachable. This exact `off-by-one` mistake will reappear constantly once you reach arrays and loops in later units — this is the first time you're meeting it, not the last.

---

## Part 5 — Simulating the rush

Everything from this lesson, combined: a rush-hour simulator for Byte-Sized Boba. Each simulated hour gets a random number of customers, and the program tracks the busiest and slowest hour as it goes.

In [ ]:
public class RushHourSimulator {
    public static void main(String[] args) {
        int totalCustomers = 0;
        int busiestHour  = Integer.MIN_VALUE;
        int slowestHour  = Integer.MAX_VALUE;

        System.out.println("=== BYTE-SIZED BOBA -- Simulated 8-Hour Shift ===");

        for (int hour = 1; hour <= 8; hour++) {
            // Each hour brings somewhere between 5 and 25 customers, inclusive
            int customers = (int)(Math.random() * (25 - 5 + 1)) + 5;

            System.out.println("Hour " + hour + ": " + customers + " customers");

            totalCustomers += customers;
            if (customers > busiestHour) busiestHour = customers;
            if (customers < slowestHour) slowestHour = customers;
        }

        System.out.println();
        System.out.println("Total for the shift: " + totalCustomers + " customers");
        System.out.println("Busiest single hour: " + busiestHour);
        System.out.println("Slowest single hour: " + slowestHour);
        System.out.println("Re-run this cell -- every shift will be different.");
    }
}

RushHourSimulator.main(null);

No two runs of that cell will ever look the same, and yet every run obeys the same rules: 8 hours, each between 5 and 25 customers. That's the trade `Math.random()` offers — you give up the ability to predict the exact output, and in exchange you get a program that can stand in for something genuinely unpredictable.

---

## Part 6 — Beyond the exam: `round`, `ceil`, `floor`, `min`, `max`

Not tested on the AP exam, but common enough in real Java that you should recognize them.

| Method | Effect | Example |
|---|---|---|
| `Math.round(double x)` | Rounds to the nearest whole number (returns `long`) | `Math.round(4.4)` → `4`, `Math.round(4.5)` → `5` |
| `Math.ceil(double x)` | Rounds **up** to the next whole number, as a `double` | `Math.ceil(4.1)` → `5.0` |
| `Math.floor(double x)` | Rounds **down** to the next whole number, as a `double` | `Math.floor(4.9)` → `4.0` |
| `Math.max(a, b)` | The larger of two values | `Math.max(3, 7)` → `7` |
| `Math.min(a, b)` | The smaller of two values | `Math.min(3, 7)` → `3` |

`Math.round` is worth noticing in particular: it does **actual rounding**, unlike a `(int)` cast, which only ever truncates. It quietly replaces the `+ 0.5` manual rounding trick from Lesson 1.5 — but since it's off the official exam list, you should still know that trick by hand.

In [ ]:
public class BeyondTheExam {
    public static void main(String[] args) {
        System.out.println("round(4.4):  " + Math.round(4.4));
        System.out.println("round(4.5):  " + Math.round(4.5));
        System.out.println("ceil(4.1):   " + Math.ceil(4.1));
        System.out.println("floor(4.9):  " + Math.floor(4.9));
        System.out.println("max(3, 7):   " + Math.max(3, 7));
        System.out.println("min(3, 7):   " + Math.min(3, 7));
    }
}

BeyondTheExam.main(null);

---

# Practice: Running the Rush

Four tasks, in order.

---

## Hack 1 — Quick review

Fill in your predictions **before** running. These reuse Lessons 1.5, 1.7, and 1.10 — a warm-up before the new material.

| # | Expression | Your prediction |
|---|---|---|
| 1 | `Math.abs(-15)` | |
| 2 | `Math.abs(-15.0)` | |
| 3 | `Math.pow(3, 3)` | |
| 4 | `Math.sqrt(100)` | |
| 5 | `(int) Math.pow(3, 3)` | |

In [ ]:
public class QuickReview {
    public static void main(String[] args) {
        System.out.println("1. " + Math.abs(-15));
        System.out.println("2. " + Math.abs(-15.0));
        System.out.println("3. " + Math.pow(3, 3));
        System.out.println("4. " + Math.sqrt(100));
        System.out.println("5. " + (int) Math.pow(3, 3));
    }
}

QuickReview.main(null);

<details>
<summary><b>Explanations</b></summary>

1. `15` — `abs(int)` returns an `int`.
2. `15.0` — `abs(double)` returns a `double`, so the `.0` sticks around even though the value is a whole number.
3. `27.0` — `pow` **always** returns a `double`, no matter what.
4. `10.0` — `sqrt` **always** returns a `double` too.
5. `27` — casting the `double` result of `pow` down to an `int` truncates the `.0` away entirely.
</details>

---

## Hack 2 — Build three random-range formulas

Write the correct inclusive-range formula for each scenario, then verify each one with a large trial loop like the ones in Part 3.

1. A coin flip: `0` for tails, `1` for heads.
2. A cup size choice: `1`, `2`, or `3` (small, medium, large).
3. A random discount percentage from `10` to `50`, in whole numbers.

For each, run **at least 100,000** trials and print the observed minimum and maximum.

In [ ]:
public class RandomFormulas {
    public static void main(String[] args) {

        // TODO 1: coin flip (0 or 1) -- run 100,000 trials, print observed min/max

        // TODO 2: cup size (1, 2, or 3) -- run 100,000 trials, print observed min/max

        // TODO 3: discount percent (10 to 50) -- run 100,000 trials, print observed min/max

    }
}

RandomFormulas.main(null);

<details>
<summary><b>One possible solution</b></summary>

```java
public class RandomFormulas {
    public static void main(String[] args) {

        // 1. Coin flip: 0 or 1
        int coinMin = Integer.MAX_VALUE, coinMax = Integer.MIN_VALUE;
        for (int i = 0; i < 100000; i++) {
            int flip = (int)(Math.random() * (1 - 0 + 1)) + 0;
            if (flip < coinMin) coinMin = flip;
            if (flip > coinMax) coinMax = flip;
        }
        System.out.println("Coin flip   -- min: " + coinMin + " max: " + coinMax);

        // 2. Cup size: 1, 2, or 3
        int sizeMin = Integer.MAX_VALUE, sizeMax = Integer.MIN_VALUE;
        for (int i = 0; i < 100000; i++) {
            int size = (int)(Math.random() * (3 - 1 + 1)) + 1;
            if (size < sizeMin) sizeMin = size;
            if (size > sizeMax) sizeMax = size;
        }
        System.out.println("Cup size    -- min: " + sizeMin + " max: " + sizeMax);

        // 3. Discount percent: 10 to 50
        int discMin = Integer.MAX_VALUE, discMax = Integer.MIN_VALUE;
        for (int i = 0; i < 100000; i++) {
            int discount = (int)(Math.random() * (50 - 10 + 1)) + 10;
            if (discount < discMin) discMin = discount;
            if (discount > discMax) discMax = discount;
        }
        System.out.println("Discount %  -- min: " + discMin + " max: " + discMax);
    }
}

RandomFormulas.main(null);
```

Expected: `min: 0 max: 1`, `min: 1 max: 3`, `min: 10 max: 50` — every formula reaching both of its own boundaries.
</details>

---

## Hack 3 — Find the missing prize

The cafe runs a "spin the wheel" promotion: every spin should land on a discount of **10, 20, 30, 40, or 50** percent, with equal odds.

The cell below has an off-by-one bug. Run it first — notice that one prize never comes up no matter how many spins you take. Fix the formula, then confirm all five values appear.

In [ ]:
public class BrokenWheel {
    public static void main(String[] args) {
        int min = 10;
        int max = 50;
        int step = 10;             // prizes come in steps of 10: 10, 20, 30, 40, 50

        boolean[] seen = new boolean[6];   // index 1-5 will represent which "step number" showed up

        for (int i = 0; i < 200000; i++) {
            // BUG: this only ever lands on 10, 20, 30, or 40 -- 50 is impossible
            int prize = ((int)(Math.random() * ((max - min) / step)) + 1) * step;
            int stepNumber = prize / step;
            seen[stepNumber] = true;
        }

        for (int s = 1; s <= 5; s++) {
            System.out.println((s * step) + "% discount landed at least once? " + seen[s]);
        }
    }
}

BrokenWheel.main(null);

> This one uses a boolean array (`boolean[] seen`) to track which prizes have shown up. Arrays are the subject of **Unit 4** — you don't need to understand the syntax deeply here, just read `seen[s]` as "did prize number `s` ever come up?"

**Your bug report** (double-click to edit):

What's missing, and what's your fixed formula?

<details>
<summary><b>Check your answer</b></summary>

There are 5 possible prizes (10, 20, 30, 40, 50), but `(max - min) / step` is `(50 - 10) / 10 = 4`, giving only 4 possible step-numbers instead of 5. The classic missing `+ 1`, hiding inside a slightly more disguised formula this time.

Fix:

```java
int prize = ((int)(Math.random() * ((max - min) / step + 1)) + 1) * step;
```

Now `(50 - 10) / 10 + 1 = 5`, and all five prizes, including `50`, become reachable. Re-run and every line should print `true`.

The lesson here is the general one from Part 4: **any time a range is built from subtraction, check whether the top end needs a `+ 1` to actually be included** — even when the formula is dressed up with extra steps like this one.
</details>

---

## Hack 4 — Extend the rush-hour simulator

Starting from the `RushHourSimulator` in Part 5, add the following:

1. Simulate a **10-hour** shift instead of 8.
2. Each hour, also randomly decide whether a **rare $50 catering order** came in — should happen roughly **1 time in 20** hours (hint: a random int from 1 to 20, and treat landing on exactly `1` as "yes").
3. Track and print how many of the 10 hours had a catering order.
4. Print the average customers per hour, as a `double` (careful — this is the integer division trap from Lesson 1.3/1.5).

In [ ]:
public class ExtendedRush {
    public static void main(String[] args) {

        // TODO 1: total customers, busiest/slowest hour, catering order counter -- set up variables

        // TODO 2: loop over 10 hours

            // TODO 3: random customers between 5 and 25 (inclusive)

            // TODO 4: random int from 1 to 20; treat landing on exactly 1 as a catering order

            // TODO 5: print this hour's results

        // TODO 6: print totals, including average customers per hour AS A DOUBLE

    }
}

ExtendedRush.main(null);

<details>
<summary><b>One possible solution</b></summary>

```java
public class ExtendedRush {
    public static void main(String[] args) {
        int totalCustomers = 0;
        int busiestHour = Integer.MIN_VALUE;
        int slowestHour = Integer.MAX_VALUE;
        int cateringOrders = 0;

        for (int hour = 1; hour <= 10; hour++) {
            int customers = (int)(Math.random() * (25 - 5 + 1)) + 5;
            int cateringRoll = (int)(Math.random() * (20 - 1 + 1)) + 1;
            boolean cateringOrder = (cateringRoll == 1);

            System.out.println("Hour " + hour + ": " + customers + " customers"
                + (cateringOrder ? "  -- CATERING ORDER!" : ""));

            totalCustomers += customers;
            if (customers > busiestHour) busiestHour = customers;
            if (customers < slowestHour) slowestHour = customers;
            if (cateringOrder) cateringOrders++;
        }

        System.out.println();
        System.out.println("Total customers:     " + totalCustomers);
        System.out.println("Busiest hour:         " + busiestHour);
        System.out.println("Slowest hour:         " + slowestHour);
        System.out.println("Catering orders:      " + cateringOrders);
        System.out.println("Average per hour:     " + (totalCustomers / 10.0));
    }
}

ExtendedRush.main(null);
```

`totalCustomers / 10.0` is the important line — dividing by `10` instead of `10.0` would silently truncate the average, exactly the Lesson 1.3 trap resurfacing one more time.

Since a catering order only lands about 1 time in 20, don't be surprised if a single 10-hour run has zero. Re-run a few times and you'll eventually see one.
</details>

---

# Self-Check: AP-style questions

**1.** What values can `Math.random()` return?

&nbsp;&nbsp;(A) Any `double` from `0.0` to `1.0`, inclusive on both ends
&nbsp;&nbsp;(B) Any `double` `>= 0.0` and `< 1.0`
&nbsp;&nbsp;(C) Any `int` from `0` to `1`
&nbsp;&nbsp;(D) Any `double` `> 0.0` and `<= 1.0`

<details><summary>Answer</summary>

**(B)**. The range is half-open: `0.0` is a possible result, but `1.0` never is.
</details>

---

**2.** Which expression produces a random integer from `1` to `10`, inclusive?

&nbsp;&nbsp;(A) `(int)(Math.random() * 10)`
&nbsp;&nbsp;(B) `(int)(Math.random() * 10) + 1`
&nbsp;&nbsp;(C) `(int)(Math.random() * 11) + 1`
&nbsp;&nbsp;(D) `(int)(Math.random() * 9) + 1`

<details>
<summary><b>Answer</b></summary>

**(B)**. `Math.random() * 10` gives a value in `[0.0, 10.0)`, which truncates to `{0, ..., 9}`, and adding `1` shifts that to `{1, ..., 10}`. (A) produces `0` through `9`. (C) can reach `11`, one too high. (D) can only reach `9`.
</details>

---

**3.** A programmer writes `(int)(Math.random() * (max - min)) + min` to generate a value from `min` to `max` inclusive. What is wrong?

&nbsp;&nbsp;(A) Nothing — this correctly includes both `min` and `max`
&nbsp;&nbsp;(B) `max` can never be produced, because the multiplier is one too small
&nbsp;&nbsp;(C) `min` can never be produced
&nbsp;&nbsp;(D) The program will throw a run-time exception

<details>
<summary><b>Answer</b></summary>

**(B)**. The multiplier should be `(max - min + 1)`. Without the `+ 1`, the largest possible value after truncation and shifting is `max - 1`, so `max` is mathematically unreachable — with no error message of any kind.
</details>

---

**4.** What does `Math.pow(2, 5)` return, and as what type?

&nbsp;&nbsp;(A) `32`, an `int`
&nbsp;&nbsp;(B) `32.0`, a `double`
&nbsp;&nbsp;(C) `10.0`, a `double`
&nbsp;&nbsp;(D) `32`, a `double`

<details>
<summary><b>Answer</b></summary>

**(B)**. `Math.pow` always returns a `double`, regardless of whether its arguments are whole numbers. (D) has the right value but the wrong printed form — a `double` result of 32 prints as `32.0`, not `32`.
</details>

---

**5.** A program calls `Math.random()` 1,000 times in a loop and checks whether every result is `< 1.0`. What is the expected outcome?

&nbsp;&nbsp;(A) All 1,000 pass, since `1.0` is documented as impossible
&nbsp;&nbsp;(B) Roughly half fail
&nbsp;&nbsp;(C) Exactly one is likely to equal `1.0`
&nbsp;&nbsp;(D) The check will crash, since `Math.random()` cannot be called in a loop

<details>
<summary><b>Answer</b></summary>

**(A)**. `Math.random()`'s documented range guarantees every result is strictly less than `1.0`, so a check across any number of calls should always pass.
</details>

---

**6.** Which best explains why a bug in a random-range formula can be dangerous in practice?

&nbsp;&nbsp;(A) It always causes an immediate crash, so it's easy to catch
&nbsp;&nbsp;(B) The compiler detects and rejects incorrect range formulas
&nbsp;&nbsp;(C) It can silently make one outcome impossible while every run still looks plausible
&nbsp;&nbsp;(D) Java automatically corrects off-by-one errors in range calculations

<details>
<summary><b>Answer</b></summary>

**(C)**. The formula compiles, runs, and produces plausible-looking output every time — it just quietly excludes one legitimate value. Only a large number of trials, or careful review of the formula itself, will reveal it.
</details>

---

**7.** Which of the following is **not** on the AP Java Quick Reference for the `Math` class?

&nbsp;&nbsp;(A) `Math.sqrt` &nbsp;&nbsp; (B) `Math.pow` &nbsp;&nbsp; (C) `Math.round` &nbsp;&nbsp; (D) `Math.random`

<details>
<summary><b>Answer</b></summary>

**(C)**. `Math.round`, along with `ceil`, `floor`, `min`, and `max`, are real and commonly used, but they are not part of the official AP-tested subset. `sqrt`, `pow`, `abs`, and `random` are the ones the exam reference sheet actually lists.
</details>

---

# Closing time

### Vocabulary to know cold

| Term | One-line definition |
|---|---|
| `Math.random()` | Returns a `double` `>= 0.0` and `< 1.0` |
| Half-open interval | A range including its lower bound but excluding its upper bound |
| Scaling | Multiplying `Math.random()` by `n` to widen the range to `[0.0, n)` |
| Shifting | Adding `min` after casting, to move the range's floor |
| Off-by-one error | A range calculation that excludes one value it should include |

### The formula to memorize

```
   (int)(Math.random() * (max - min + 1)) + min
```

### The five things that will show up on the exam

1. `Math.random()` returns `[0.0, 1.0)` — the top end is **never** reachable.
2. `Math.pow` and `Math.sqrt` **always** return a `double`.
3. Scale, then cast, then shift — in that order.
4. Forgetting `+ 1` makes the maximum value **impossible**, silently.
5. `round`, `ceil`, `floor`, `min`, `max` are real but **not** on the AP reference sheet.

### Before you submit, check that you:

- [ ] Ran every code cell top to bottom
- [ ] Predicted all five expressions in Hack 1 before running it
- [ ] Wrote and statistically verified all three formulas in Hack 2
- [ ] Found and fixed the off-by-one bug in Hack 3, confirming all five prizes appear
- [ ] Extended the simulator in Hack 4, including the `/ 10.0` average
- [ ] Attempted all seven self-check questions before revealing answers

### Next up

**1.12 — Objects: Instances of Classes.** Every program in this series has lived entirely inside `static` methods — no object has ever been built. That changes now: you'll create your first real object, a `Scanner` or a `String` with its own identity separate from the class that defines it, and find out what `new` has been quietly promising this whole time.

See you at the next shift 🧋